# Lekcija 12 - Smanjenje povijesti chata uz Agent Scratchpad

Ovaj bilježnik prikazuje kako upravljati kontekstom u dugim razgovorima koristeći Microsoft Agent Framework. Kako razgovori rastu, broj tokena se povećava — na kraju premašujući kontekstni prozor modela. To rješavamo uz pomoć **uzorka sažimanja konteksta** i **agentnog scratchpada** za trajnu memoriju.

## Što ćete naučiti:
1. **Zašto je upravljanje kontekstom važno**: Razumijevanje ograničenja tokena i kontekstnih prozora
2. **Agenti svjesni konteksta**: Izgradnja agenata koji upravljaju vlastitim kontekstom razgovora
3. **Uzorak sažimanja konteksta**: Korištenje alata za sažimanje povijesti razgovora
4. **Agent Scratchpad**: Trajna memorija koja preživljava smanjenje konteksta

## Preduvjeti:
- Postavljen Azure OpenAI s konfiguriranim varijablama okoline
- Razumijevanje osnovnih pojmova o agentima iz prethodnih lekcija


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Zašto je upravljanje kontekstom važno

Svaki LLM ima ograničen **prozor konteksta** — maksimalan broj tokena koje može obraditi u jednom zahtjevu. Kako se višerunda razgovora odvija:

- **Broj tokena linearno raste** sa svakom korisničkom porukom i odgovorom asistenta.
- **Tokeni za upit dominiraju cijenom** jer se cjelokupna povijest ponovo šalje svaki put.
- Na kraju, razgovor **prelazi prozor konteksta** i model ili skraćuje tekst ili baca pogrešku.

### Strategije za upravljanje kontekstom

| Strategija | Kako funkcionira | Kompromis |
|---|---|---|
| **Skraćivanje** | Izbacivanje najstarijih poruka | Gubi se rani kontekst |
| **Sažimanje** | Sažimanje starijih poruka u sažetak | Gubi se dio detalja, ali se zadržavaju ključne točke |
| **Radni blok / Vanjska memorija** | Spremanje ključnih činjenica izvan razgovora | Zahtijeva pozive alata, ali preživljava bilo kakvo smanjenje |

U ovom bilježniku kombiniramo **sažimanje** s **radnim blokom** kako bi agent mogao održavati kontinuitet čak i kada se povijest razgovora sažima.


## Izrada agenta svjesnog konteksta


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulacija dugog razgovora

Prođimo kroz razgovor s više poteza kako bismo vidjeli kako se kontekst akumulira. Agent bi trebao zadržati ključne detalje (preferencije, budžet, datume putovanja) tijekom razgovora i pokazati kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Primijetite kako agent zadržava kontekst iz ranijih dijelova razgovora — zna o Japanu, sushiju, hramovima, fotografiji, proračunu od 3000 dolara, samostalnom putovanju i vremenskom razdoblju u travnju. U kratkom razgovoru to dobro funkcionira, ali kako razgovor raste, puni povijesni zapis postaje skup za ponovno slanje.

Nastavimo razgovor s više dijaloga kako bismo vidjeli nakupljanje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Obrasac sažimanja konteksta

Kako razgovor raste, možemo koristiti **alat za sažimanje** za sažimavanje prikupljenog konteksta u sažeti format. Agent poziva ovaj alat kako bi zabilježio ključne preferencije tako da, čak i ako se starije poruke uklone, bitne informacije budu sačuvane.

Ovaj obrazac je temelj za sofisticiranije smanjenje povijesti:
1. Agent prepoznaje ključne činjenice iz razgovora
2. Poziva alat za sažimanje da ih sačuva
3. Starije poruke se mogu sigurno ukloniti jer sažetak sadrži ono što je važno

Ispod definiramo alat `summarize_preferences` koji agent može pozvati kako bi zabilježio sažeti pregled onoga što je naučio.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sažetak

U ovoj lekciji naučili ste kako upravljati kontekstom u dugotrajnim razgovorima agenata koristeći Microsoft Agent Framework:

### Ključni pojmovi
- **Prozori konteksta su ograničeni** — svaki token u povijesti razgovora košta i računa se prema ograničenju.
- **Alati za sažimanje** omogućuju agentu da kondenzira prikupljeni kontekst u kompaktne sažetke, smanjujući korištenje tokena uz očuvanje bitnih informacija.
- **Agenti s bilješkama** pružaju trajnu vanjsku memoriju koja preživljava svako smanjenje razgovora.

### Što ste izgradili
- **Agent svjestan konteksta** koji održava kontinuitet kroz višekratne razgovore
- **Alat za sažimanje** (`summarize_preferences`) koji bilježi ključne korisničke detalje u kompaktni format
- **Višekratni razgovor** koji demonstrira zadržavanje konteksta i rukovanje promjenama

### Primjene u stvarnom svijetu
- **Botovi za korisničku podršku**: Pamte preferencije kroz duge sesije podrške
- **Osobni asistenti**: Prate tekuće projekte bez ponovnog objašnjavanja konteksta
- **Obrazovni tutori**: Održavaju napredak učenika kroz mnoge interakcije

### Sljedeći koraci
- Implementirati puni alat za bilješke s trajnom pohranom na temelju datoteka
- Dodati automatsko skraćivanje povijesti nakon sažimanja
- Kombinirati s vektorskim bazama podataka za semantičko pretraživanje memorije
- Izgraditi agente koji mogu nastaviti razgovore danima kasnije s punim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o odricanju odgovornosti**:
Ovaj je dokument preveden korištenjem AI prevodilačke usluge [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo osigurati točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati službenim i autoritativnim izvorom. Za važne informacije preporučuje se profesionalni prijevod od strane ljudskog stručnjaka. Ne snosimo odgovornost za bilo kakva nesporazuma ili kriva tumačenja nastala korištenjem ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
